# CNN Chest X-Ray Classifier with MLflow Model Versioning
Trains the same CNN architecture with **Adam** and **SGD** optimizers, logs every run to MLflow, registers each model version, and automatically aliases the best one as `@champion`.

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import mlflow
import mlflow.tensorflow
from mlflow import MlflowClient
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    f1_score, accuracy_score, precision_score, recall_score
)

## Configuration
Edit the paths and hyper-parameters here before running.

In [7]:
TRAIN_DIR   = '/home/arshia/Downloads/archive/train'
VAL_DIR     = '/home/arshia/Downloads/archive/val'
TEST_DIR    = '/home/arshia/Downloads/archive/test'
EPOCHS      = 2
BATCH_SIZE  = 32
IMG_SIZE    = (224, 224)
MODEL_NAME  = "chest-xray-cnn"   # MLflow Registered Model name

# ── Optimizers to compare ──────────────────────────────────────────────────
# Add / remove entries here to run more experiments
OPTIMIZER_CONFIGS = [
    {
        "name": "adam",
        "optimizer": tf.keras.optimizers.Adam(learning_rate=1e-3),
        "description": "Adam optimizer, lr=1e-3"
    },
    {
        "name": "sgd",
        "optimizer": tf.keras.optimizers.SGD(learning_rate=1e-2, momentum=0.9),
        "description": "SGD optimizer, lr=1e-2, momentum=0.9"
    },
]

## Data 

In [8]:
def build_generators():
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=10,
        zoom_range=0.1,
        horizontal_flip=True
    )
    training_set = train_datagen.flow_from_directory(
        TRAIN_DIR, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='binary'
    )

    valid_datagen = ImageDataGenerator(rescale=1./255)
    validation_set = valid_datagen.flow_from_directory(
        VAL_DIR, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='binary'
    )

    test_datagen = ImageDataGenerator(rescale=1./255)
    test_set = test_datagen.flow_from_directory(
        TEST_DIR, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
    )

    return training_set, validation_set, test_set

## Model

In [9]:
def build_model(optimizer):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv2D(32, kernel_size=3, activation='relu',
                               input_shape=(224, 224, 3)),
        tf.keras.layers.MaxPool2D(pool_size=2, strides=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

## Accuracy

In [10]:
def evaluate_model(model, test_set):
    y_prob = model.predict(test_set).flatten()
    y_pred = (y_prob > 0.5).astype(int)
    y_true = test_set.classes

    metrics = {
        "test_accuracy":  accuracy_score(y_true, y_pred),
        "test_precision": precision_score(y_true, y_pred, zero_division=0),
        "test_recall":    recall_score(y_true, y_pred, zero_division=0),
        "test_f1":        f1_score(y_true, y_pred, zero_division=0),
        "test_roc_auc":   roc_auc_score(y_true, y_prob),
    }

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:\n", cm)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred,
                                target_names=test_set.class_indices.keys()))

    return metrics, y_true, y_prob, cm

## MLflow Experiment Loop
Runs one MLflow **Run** per optimizer, registers the model, and tags each version.

In [11]:
mlflow.set_experiment("chest-xray-cnn-comparison")
client = MlflowClient()

# Create the registered model (idempotent)
try:
    client.create_registered_model(
        MODEL_NAME,
        description="Binary chest X-ray classifier (pneumonia vs normal)"
    )
    print(f"Created registered model: {MODEL_NAME}")
except mlflow.exceptions.MlflowException:
    print(f"Registered model '{MODEL_NAME}' already exists – reusing it.")

training_set, validation_set, test_set = build_generators()
run_ids = {}

for cfg in OPTIMIZER_CONFIGS:
    opt_name = cfg["name"]
    print(f"\n{'='*55}")
    print(f"  Training with optimizer: {opt_name.upper()}")
    print(f"{'='*55}")

    with mlflow.start_run(run_name=f"cnn_{opt_name}") as run:
        run_id = run.info.run_id
        run_ids[opt_name] = run_id

        # Log hyper-parameters
        mlflow.log_params({
            "optimizer":    opt_name,
            "epochs":       EPOCHS,
            "batch_size":   BATCH_SIZE,
            "image_size":   f"{IMG_SIZE[0]}x{IMG_SIZE[1]}",
            "dropout":      0.5,
            "conv_filters": 32,
            "dense_units":  128,
        })
        mlflow.set_tag("description", cfg["description"])

        # Build model
        model = build_model(cfg["optimizer"])

        # Keras callback to log per-epoch metrics
        class MLflowEpochLogger(tf.keras.callbacks.Callback):
            def on_epoch_end(self, epoch, logs=None):
                logs = logs or {}
                mlflow.log_metrics({
                    "epoch_train_loss":     logs.get("loss", 0),
                    "epoch_train_accuracy": logs.get("accuracy", 0),
                    "epoch_val_loss":       logs.get("val_loss", 0),
                    "epoch_val_accuracy":   logs.get("val_accuracy", 0),
                }, step=epoch)

        history = model.fit(
            training_set,
            validation_data=validation_set,
            epochs=EPOCHS,
            callbacks=[MLflowEpochLogger()]
        )

        # Evaluate
        test_loss, test_acc = model.evaluate(test_set)
        mlflow.log_metric("test_loss", test_loss)

        metrics, y_true, y_prob, cm = evaluate_model(model, test_set)
        mlflow.log_metrics(metrics)

        
        # Register model version
        mlflow.tensorflow.log_model(
            model,
            artifact_path="model",
            registered_model_name=MODEL_NAME,
        )

        # Tag the new version
        versions = client.search_model_versions(
            f"name='{MODEL_NAME}' and run_id='{run_id}'"
        )
        if versions:
            mv = versions[0]
            client.set_model_version_tag(MODEL_NAME, mv.version, "optimizer", opt_name)
            client.update_model_version(
                MODEL_NAME, mv.version,
                description=(
                    f"{cfg['description']} | "
                    f"F1={metrics['test_f1']:.4f} | "
                    f"AUC={metrics['test_roc_auc']:.4f}"
                )
            )
            print(f"\n✓ Registered as '{MODEL_NAME}' version {mv.version}")

Registered model 'chest-xray-cnn' already exists – reusing it.
Found 6800 images belonging to 2 classes.
Found 1620 images belonging to 2 classes.
Found 110 images belonging to 2 classes.


2026/04/11 10:05:29 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet




  Training with optimizer: ADAM


/home/arshia/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/2


I0000 00:00:1775916330.531076    4728 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


213/213 ━━━━━━━━━━━━━━━━━━━━ 230s 1s/step - accuracy: 0.8363 - loss: 1.0505 - val_accuracy: 0.9432 - val_loss: 0.1613
Epoch 2/2
213/213 ━━━━━━━━━━━━━━━━━━━━ 224s 1s/step - accuracy: 0.9137 - loss: 0.2247 - val_accuracy: 0.9500 - val_loss: 0.1566
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 324ms/step - accuracy: 0.9000 - loss: 0.2136
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 326ms/step

Confusion Matrix:
 [[52  3]
 [ 8 47]]

Classification Report:
              precision    recall  f1-score   support

      NORMAL       0.87      0.95      0.90        55
   PNEUMONIA       0.94      0.85      0.90        55

    accuracy                           0.90       110
   macro avg       0.90      0.90      0.90       110
weighted avg       0.90      0.90      0.90       110



NameError: name 'plot_training_curves' is not defined

## Compare Versions & Promote Best Model

In [12]:
print("\n" + "="*55)
print("  Comparing runs and promoting best model …")
print("="*55)

best_f1      = -1
best_version = None

for version in client.search_model_versions(f"name='{MODEL_NAME}'"):
    run  = client.get_run(version.run_id)
    f1   = run.data.metrics.get("test_f1", 0)
    print(f"  Version {version.version} | "
          f"optimizer={version.tags.get('optimizer','?')} | "
          f"F1={f1:.4f}")
    if f1 > best_f1:
        best_f1      = f1
        best_version = version.version

if best_version:
    client.set_registered_model_alias(MODEL_NAME, "champion", best_version)
    print(f"\n🏆  Version {best_version} aliased as 'champion' (F1={best_f1:.4f})")


  Comparing runs and promoting best model …
  Version 2 | optimizer=sgd | F1=0.8713
  Version 1 | optimizer=adam | F1=0.9057

🏆  Version 1 aliased as 'champion' (F1=0.9057)


In [13]:
champion = mlflow.tensorflow.load_model(f"models:/{MODEL_NAME}@champion")
champion.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 394272)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    50,466,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 151,403,909 (577.56 MB)

 Trainable params: 50,467,969 (192.52 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 100,935,940 (385.04 MB)

In [18]:
keras_path = f"cnn_{opt_name}.keras"  # saves in current working directory
model.save(keras_path)
mlflow.log_artifact(keras_path, artifact_path="keras_model")